# AFRICA GIANTS â€” Continuous Training on Kaggle

Fine-tunes **McGill-NLP/AfriqueLlama-8B** (Llama 3.1 8B, 20 African languages incl. Swahili)
on Tanzanian business/regulatory data.

- **T4 / V100 / A100 (sm_70+):** Unsloth QLoRA â€” 2Ã— faster training
- **P100 (sm_60):** standard BitsAndBytes QLoRA with PyTorch cu118

In [ ]:
# ── Step 1: GPU detection BEFORE importing torch ─────────────────────────────
# nvidia-smi never fails on P100/T4 so we use it to avoid the PyTorch
# sm_60 import warning that fires before we can catch it.
import os, subprocess

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"   # synchronous CUDA errors
os.environ["TORCH_USE_CUDA_DSA"]   = "1"   # device-side assertion details

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,compute_cap,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True
)
if smi.returncode != 0:
    raise RuntimeError("nvidia-smi failed -- no GPU available. Enable GPU in Kaggle -> Settings -> Accelerator.")

parts    = [p.strip() for p in smi.stdout.strip().split(",")]
GPU_NAME = parts[0]                      # e.g. "Tesla P100-PCIE-16GB"
SM       = int(float(parts[1]) * 10)    # "6.0" -> 60, "7.5" -> 75
VRAM_GB  = parts[2]                      # e.g. "16160 MiB"

USE_UNSLOTH = SM >= 70   # T4=75, V100=70, A100=80 pass; P100=60 falls back

print(f"GPU         : {GPU_NAME}")
print(f"VRAM        : {VRAM_GB}")
print(f"Compute     : sm_{SM}")
print(f"Training via: {'Unsloth QLoRA (fast path)' if USE_UNSLOTH else 'BitsAndBytes QLoRA (P100 compat path)'}")

if SM < 70:
    print(f"[gpu-check] WARNING: {GPU_NAME} (sm_{SM}) assigned -- attempting BitsAndBytes P100 compat path.")
    print("[gpu-check] T4 x2 preferred; if training fails, check Kaggle Settings -> Accelerator -> GPU T4 x2.")

In [ ]:
# ── Step 2: Install the right stack for the assigned GPU ──────────────────────
print(f"[install] GPU path: {'Unsloth' if USE_UNSLOTH else 'BitsAndBytes P100'}")

if USE_UNSLOTH:
    # T4/V100/A100 – Unsloth prebuilt wheel, no CUDA compilation
    ret = get_ipython().system("pip install -q unsloth 2>&1 | tail -3")
    print(f"[install] unsloth done (exit {ret})")
else:
    # P100 (sm_60): no torch wheel supports sm_60 with CUDA 11.8+ (earliest cu118 is 2.2.0+cu118, requires sm_70+).
    # Skip torch install; pre-installed torch 2.10+cu128 will emit a CUDA capability warning but
    # we still install the BitsAndBytes stack so the imports cell can run.
    print("[install] WARNING: No torch wheel supports P100 sm_60 -- T4 must be assigned for GPU training.")
    get_ipython().system("pip install -q 'transformers>=4.43.0' peft accelerate bitsandbytes 'trl>=0.10,<0.16' 2>&1 | tail -3")
    print("[install] BitsAndBytes stack done")

get_ipython().system("pip install -q datasets huggingface_hub 2>&1 | tail -3")
print("[install] datasets/huggingface_hub done")

# Report installed versions so we can see them in logs
import subprocess, sys
for pkg in ["unsloth", "trl", "transformers", "torch", "peft", "bitsandbytes"]:
    r = subprocess.run([sys.executable, "-m", "pip", "show", pkg],
                       capture_output=True, text=True)
    for line in r.stdout.splitlines():
        if line.startswith("Version:"):
            print(f"  {pkg}: {line}")
            break
    else:
        print(f"  {pkg}: not installed")

print("[install] complete")

In [ ]:
# ── Step 3: Imports – conditional on GPU path ───────────────────────────────────
print("[imports] starting ...")

import os, torch
print(f"[imports] torch {torch.__version__}")

from datasets import load_dataset
from transformers import TrainingArguments
print("[imports] transformers/datasets OK")

from trl import SFTTrainer
try:
    from trl import SFTConfig
    print("[imports] trl SFTTrainer + SFTConfig OK")
except ImportError:
    SFTConfig = None
    print("[imports] trl SFTTrainer OK (SFTConfig not available, falling back to TrainingArguments)")

from huggingface_hub import HfApi, create_repo, login, whoami
print("[imports] huggingface_hub OK")

if USE_UNSLOTH:
    print("[imports] loading unsloth ...")
    from unsloth import FastLanguageModel, is_bfloat16_supported
    BF16 = is_bfloat16_supported()
    print("[imports] unsloth OK")
else:
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    BF16 = False
    print("[imports] BitsAndBytes path imports OK")

print(f"[imports] torch      : {torch.__version__}")
print(f"[imports] GPU        : {torch.cuda.get_device_name(0)}  sm_{SM}")
print(f"[imports] BF16       : {BF16}")
print("[imports] done")

In [ ]:
# HF login – Kaggle secret label must be: AFRICA_GIANTS
print("[secrets] loading HF token ...")
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("AFRICA_GIANTS")
print("[secrets] token loaded, logging in ...")
login(token=hf_token)
me = whoami(token=hf_token)["name"]
print(f"[secrets] logged in as: {me}")

In [ ]:
# ── Hardcoded repo references ──────────────────────────────────────────────────
BASE_MODEL        = "McGill-NLP/AfriqueLlama-8B"
ADAPTER_REPO      = "prospaprospa007/africa-giants-adapter-v1"
MERGED_MODEL_REPO = "prospaprospa007/africa-giants-model-v1"
DATASET_REPO      = "prospaprospa007/africa-giants-dataset"

SMOKE_TEST     = True
MAX_SEQ_LENGTH = 512 if SMOKE_TEST else 2048
LOSS_THRESHOLD = 2.5

print(f"[config] SMOKE_TEST={SMOKE_TEST}  MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}")
print(f"[config] BASE_MODEL={BASE_MODEL}")
print(f"[config] DATASET_REPO={DATASET_REPO}")
print(f"[config] ADAPTER_REPO={ADAPTER_REPO}")

api = HfApi(token=hf_token)
for repo_id, repo_type in [
    (ADAPTER_REPO, "model"), (MERGED_MODEL_REPO, "model"), (DATASET_REPO, "dataset")
]:
    create_repo(repo_id=repo_id, repo_type=repo_type, private=True, exist_ok=True, token=hf_token)
    print(f"[config] ready: {repo_type} {repo_id}")

print("[config] done")

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────────
if USE_UNSLOTH:
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
        token=hf_token,
        device_map={'': torch.cuda.current_device()},
    )
    print(f"[model] eos_token_id: {tokenizer.eos_token_id}")
    model = FastLanguageModel.get_peft_model(
        model, r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        bias="none", use_gradient_checkpointing="unsloth", random_state=3407,
    )
else:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=hf_token, trust_remote_code=True)
    tokenizer.pad_token = tokenizer.eos_token
    print(f"[model] eos_token (BitsAndBytes path): {repr(tokenizer.eos_token)}")
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL, quantization_config=bnb_config,
        device_map={'': torch.cuda.current_device()},
        token=hf_token, trust_remote_code=True,
    )
    model = prepare_model_for_kbit_training(model)
    model = get_peft_model(model, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        bias="none", task_type="CAUSAL_LM",
    ))

model.print_trainable_parameters()
print(f"Loaded {BASE_MODEL} via {'Unsloth' if USE_UNSLOTH else 'BitsAndBytes'}")

# ── EOS fix: ensure eos_token is a real vocab token ──────────────────────────
_vocab = tokenizer.get_vocab()
_eos_candidates = ["</s>", "<|end_of_text|>", "<|im_end|>", "<eos>", "<|eot_id|>"]
_found = next((t for t in _eos_candidates if t in _vocab), None)
if _found and tokenizer.eos_token not in _vocab:
    tokenizer.eos_token = _found
    tokenizer.eos_token_id = _vocab[_found]
    print(f"[fix] eos_token set to: {_found}")
else:
    print(f"[fix] eos_token ok: {tokenizer.eos_token}")
print(f"[vocab] sample: {list(_vocab.keys())[:50]}")

In [ ]:
# ── Dataset ──────────────────────────────────���────────────────────────────────
raw_dataset = load_dataset(DATASET_REPO, token=hf_token)
print(raw_dataset)

SYSTEM_PROMPT = (
    "Wewe ni msaidizi wa AI wa biashara za Tanzania. "
    "Unajibu maswali kuhusu sheria za biashara, kodi, usajili wa kampuni kwa Kiswahili na Kiingereza. "
    "You are a Tanzanian business AI assistant answering questions about regulations, "
    "tax, company registration, and financial rules in Swahili and English."
)

# AfriqueLlama has its own tokenizer -- read EOS from it directly.
# Decode from eos_token_id if the string value is None or a placeholder.
_raw_eos = tokenizer.eos_token
_PLACEHOLDER_EOS = {"<EOS_TOKEN>", "<eos>", "[EOS]", ""}
if _raw_eos and _raw_eos not in _PLACEHOLDER_EOS:
    EOS_TOKEN = _raw_eos
else:
    _eos_id = tokenizer.eos_token_id
    EOS_TOKEN = tokenizer.decode([_eos_id]) if _eos_id is not None else "</s>"
    print(f"[data] WARNING: eos_token={repr(_raw_eos)} is placeholder -- "
          f"resolved via id={_eos_id} to {repr(EOS_TOKEN)}")
print(f"[data] EOS_TOKEN = {repr(EOS_TOKEN)}")

def fmt(ex):
    inst = ex.get("instruction", "")
    ctx  = ex.get("input", "") or ""
    out  = ex.get("output", "")
    user = f"Context: {ctx}\n\n{inst}" if ctx.strip() else inst
    if USE_UNSLOTH:
        msgs = [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user},
            {"role": "assistant", "content": out},
        ]
        text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
        if not text.endswith(EOS_TOKEN):
            text = text + EOS_TOKEN
    else:
        text = (
            f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
            f"{SYSTEM_PROMPT}{EOS_TOKEN}"
            f"<|start_header_id|>user<|end_header_id|>\n\n"
            f"{user}{EOS_TOKEN}"
            f"<|start_header_id|>assistant<|end_header_id|>\n\n"
            f"{out}{EOS_TOKEN}"
        )
    return {"text": text}

train_ds = raw_dataset["train"].map(fmt, batched=False)
val_src  = raw_dataset.get("validation") or raw_dataset["train"].select(range(min(10, len(raw_dataset["train"]))))
eval_ds  = val_src.map(fmt, batched=False)
print(f"Train: {len(train_ds)}  Eval: {len(eval_ds)}")

In [ ]:
# ── Train ─────────────────────────────────────────────────────────────────────────────
import inspect
import trl as _trl_mod
import transformers as _tf_mod

print(f"[train] trl={_trl_mod.__version__}  transformers={_tf_mod.__version__}")

# ── 1. Probe SFTTrainer signature ────────────────────────────────────────────────────
_sft_sig_params = set(inspect.signature(SFTTrainer.__init__).parameters)

if "processing_class" in _sft_sig_params:
    _tok_kwarg = {"processing_class": tokenizer}
    print("[train] SFTTrainer tok-kwarg: processing_class")
else:
    _tok_kwarg = {"tokenizer": tokenizer}
    print("[train] SFTTrainer tok-kwarg: tokenizer")

# ── 2. Pick args class ───────────────────────────────────────────────────────────────
_ArgsClass = SFTConfig if SFTConfig is not None else TrainingArguments
print(f"[train] args class: {_ArgsClass.__name__}")
_args_sig_params = set(inspect.signature(_ArgsClass.__init__).parameters)

_eval_key = "eval_strategy" if "eval_strategy" in _args_sig_params else "evaluation_strategy"
print(f"[train] eval kwarg: {_eval_key}")

# ── 3. Training kwargs ───────────────────────────────────────────────────────────────
_base_kwargs = {
    "output_dir": "./outputs",
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 2 if SMOKE_TEST else 4,
    "warmup_steps": 2,
    "max_steps": 10 if SMOKE_TEST else -1,
    "num_train_epochs": 1 if SMOKE_TEST else 3,
    "learning_rate": 2e-4,
    "fp16": not BF16,
    "bf16": BF16,
    "logging_steps": 1,
    "optim": "adamw_8bit" if USE_UNSLOTH else "adamw_torch",
    "weight_decay": 0.01,
    "lr_scheduler_type": "cosine",
    "seed": 3407,
    "report_to": "none",
    "save_strategy": "no",
    "eval_steps": 5,
    "dataloader_pin_memory": False,
    "gradient_checkpointing": not USE_UNSLOTH,
    _eval_key: "steps",
}
_sft_specific = {
    "dataset_text_field": "text",
    "max_seq_length": MAX_SEQ_LENGTH,
    "packing": False,
    "dataset_num_proc": 2,
}
_args_kwargs = {k: v for k, v in {**_base_kwargs, **_sft_specific}.items()
               if k in _args_sig_params}
_sft_direct  = {k: v for k, v in _sft_specific.items()
               if k not in _args_sig_params and k in _sft_sig_params}
_nowhere     = {k: v for k, v in _sft_specific.items()
               if k not in _args_sig_params and k not in _sft_sig_params}
print(f"[train] args class kwargs ({len(_args_kwargs)}): {sorted(_args_kwargs)}")
print(f"[train] SFTTrainer direct kwargs ({len(_sft_direct)}): {sorted(_sft_direct)}")
if _nowhere:
    print(f"[train] WARNING: not accepted anywhere, skipping: {sorted(_nowhere)}")

# ── 4. Build args object ─────────────────────────────────────────────────────────────
_build_args = dict(_args_kwargs)
_args_obj = None
for _attempt in range(20):
    try:
        _args_obj = _ArgsClass(**_build_args)
        print(f"[train] {_ArgsClass.__name__} built OK on attempt {_attempt + 1}")
        break
    except TypeError as _e:
        _msg = str(_e)
        _bad = _msg.split("'")[1] if _msg.count("'") >= 2 else None
        if _bad and _bad in _build_args:
            print(f"[train] WARNING: {_ArgsClass.__name__} rejected {_bad!r} -- removing and retrying")
            _build_args.pop(_bad)
        else:
            print(f"[train] FATAL: {_ArgsClass.__name__} TypeError not fixable: {_e}")
            raise
if _args_obj is None:
    raise RuntimeError("[train] Could not build args object after 20 attempts")

# ── 5a. EOS guard (double-safety): ensure eos_token is a real vocab token ───────────
_vocab = tokenizer.get_vocab()
_eos_candidates = ["</s>", "<|end_of_text|>", "<|im_end|>", "<eos>", "<|eot_id|>"]
_found = next((t for t in _eos_candidates if t in _vocab), None)
if _found and tokenizer.eos_token not in _vocab:
    tokenizer.eos_token = _found
    tokenizer.eos_token_id = _vocab[_found]
    print(f"[eos] forced: {_found}")
else:
    print(f"[eos] eos_token ok: {tokenizer.eos_token}")

# ── 5b. Build SFTTrainer ─────────────────────────────────────────────────────────────
_build_tok  = dict(_tok_kwarg)
_build_sft  = dict(_sft_direct)
_trainer = None
for _attempt in range(20):
    try:
        _trainer = SFTTrainer(
            model=model,
            train_dataset=train_ds,
            eval_dataset=eval_ds,
            args=_args_obj,
            **_build_tok,
            **_build_sft,
        )
        print(f"[train] SFTTrainer built OK on attempt {_attempt + 1}")
        break
    except TypeError as _e:
        _msg = str(_e)
        _bad = _msg.split("'")[1] if _msg.count("'") >= 2 else None
        if _bad and _bad in _build_sft:
            print(f"[train] WARNING: SFTTrainer rejected {_bad!r} -- removing and retrying")
            _build_sft.pop(_bad)
        elif _bad and _bad in _build_tok:
            if _bad == "processing_class":
                _build_tok = {"tokenizer": tokenizer}
                print("[train] WARNING: processing_class rejected, switching to tokenizer=")
            elif _bad == "tokenizer":
                _build_tok = {"processing_class": tokenizer}
                print("[train] WARNING: tokenizer= rejected, switching to processing_class=")
            else:
                print(f"[train] FATAL: SFTTrainer TypeError not fixable: {_e}")
                raise
        else:
            print(f"[train] FATAL: SFTTrainer TypeError not fixable: {_e}")
            raise
if _trainer is None:
    raise RuntimeError("[train] Could not build SFTTrainer after 20 attempts")

trainer = _trainer
print(f"[train] Starting on {GPU_NAME} via {'Unsloth' if USE_UNSLOTH else 'BitsAndBytes'} ...")
stats = trainer.train()
print(f"[train] Done. Runtime: {stats.metrics['train_runtime']:.1f}s")

In [ ]:
eval_results    = trainer.evaluate()
validation_loss = eval_results.get("eval_loss", 999.0)
gate_passed     = validation_loss <= LOSS_THRESHOLD
print(f"Val loss: {validation_loss:.4f}  threshold: {LOSS_THRESHOLD}  â†’ {'PASSED âœ“' if gate_passed else 'FAILED âœ—'}")

In [ ]:
if gate_passed:
    print(f"Pushing adapter to {ADAPTER_REPO}...")
    if USE_UNSLOTH:
        model.push_to_hub_merged(ADAPTER_REPO, tokenizer, save_method="lora", token=hf_token)
    else:
        model.push_to_hub(ADAPTER_REPO, token=hf_token)
        tokenizer.push_to_hub(ADAPTER_REPO, token=hf_token)

    card = f"""---
language:
- sw
- en
license: llama3.1
base_model: {BASE_MODEL}
tags:
- llama-3.1
- african-languages
- swahili
- tanzanian-business
- qlora
- {'unsloth' if USE_UNSLOTH else 'bitsandbytes'}
- peft
- lora
pipeline_tag: text-generation
---

# Africa Giants â€” Tanzanian Business AI (LoRA Adapter)

QLoRA fine-tune of [{BASE_MODEL}](https://huggingface.co/{BASE_MODEL})
on Tanzanian business, tax, company registration, and financial regulation data.

**Base model:** Llama 3.1 8B pre-trained on 20 African languages including Swahili.  
**Languages:** Swahili (sw), English (en)  
**Training:** QLoRA r=16 on {GPU_NAME} (sm_{SM})  
**Validation loss:** {validation_loss:.4f}
"""
    api.upload_file(
        path_or_fileobj=card.encode(), path_in_repo="README.md",
        repo_id=ADAPTER_REPO, repo_type="model", token=hf_token,
    )
    print(f"Adapter + model card pushed to {ADAPTER_REPO} âœ“")
else:
    print(f"Gate failed â€” not pushing.")

In [ ]:
MERGE_AND_PUSH = False
if MERGE_AND_PUSH and gate_passed and USE_UNSLOTH:
    model.push_to_hub_merged(MERGED_MODEL_REPO, tokenizer, save_method="merged_16bit", token=hf_token)
    print(f"Merged model â†’ {MERGED_MODEL_REPO} âœ“")
else:
    print("Skipping merged model push.")